# Train floorplan_gen v3 on Colab / Kaggle (GPU)
Two models are trained: **BoxNet** (graph -> room boxes, minutes) and **RefineNet** (boxes -> room map, the longer one).
1. Runtime / Accelerator -> **GPU**.
2. Put `floorplan_gen.zip` and your `ResPlan` folder (ResPlan.pkl + split.json) in Google Drive (Colab) or as Kaggle datasets.
3. Edit the paths in cell 2 and **Run all**. Checkpoints go to `CKPT_DIR`; re-run a training cell with `--resume` if the session dies.
4. When done, copy `box_best.pt` and `refine_best.pt` from `CKPT_DIR` into `floorplan_gen/checkpoints/` on your PC.

In [ ]:
!nvidia-smi -L
!pip -q install shapely opencv-python-headless scipy

In [ ]:
import os, zipfile, shutil, sys
# ---------- EDIT THESE ----------
ON_COLAB = 'google.colab' in sys.modules
if ON_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    PROJECT_ZIP = '/content/drive/MyDrive/floorplan_gen.zip'
    DATA_DIR    = '/content/drive/MyDrive/ResPlan'            # ResPlan.pkl + split.json
    CKPT_DIR    = '/content/drive/MyDrive/floorplan_ckpt'     # survives session end
else:  # Kaggle
    PROJECT_ZIP = '/kaggle/input/floorplan-gen/floorplan_gen.zip'
    DATA_DIR    = '/kaggle/input/resplan'
    CKPT_DIR    = '/kaggle/working/ckpt'
# --------------------------------
WORK = '/content' if ON_COLAB else '/kaggle/working'
zipfile.ZipFile(PROJECT_ZIP).extractall(WORK)
%cd {WORK}/floorplan_gen
os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
!python prepare_data.py --data-dir {DATA_DIR}

In [ ]:
# Stage 1 (fast): graph + sizes -> room boxes. Add --resume to continue.
!python train_box.py --epochs 200 --d 256 --heads 8 --layers 6 --ckpt-dir {CKPT_DIR}

In [ ]:
# Stage 2 (main cost): boxes -> room map, trained on REAL boxes. Add --resume to continue.
!python train_refine.py --epochs 40 --ckpt-dir {CKPT_DIR} --max-minutes 150

### Stage 2 fine-tune on BoxNet's own predictions (big quality boost end-to-end)

In [ ]:
!cp {CKPT_DIR}/refine_best.pt {CKPT_DIR}/refine_gt_boxes.pt
!python train_refine.py --epochs 15 --lr 5e-4 --init {CKPT_DIR}/refine_gt_boxes.pt --boxnet {CKPT_DIR}/box_best.pt --ckpt-dir {CKPT_DIR}

In [ ]:
# Learn the best-of-K scoring weights from data (writes score_weights.json next to the checkpoints)
!python calibrate_score.py --n 300 --k 8 --box-ckpt {CKPT_DIR}/box_best.pt --ref-ckpt {CKPT_DIR}/refine_best.pt

In [ ]:
!python evaluate.py --box-ckpt {CKPT_DIR}/box_best.pt --ref-ckpt {CKPT_DIR}/refine_best.pt

In [ ]:
!python generate.py --samples-dir samples --box-ckpt {CKPT_DIR}/box_best.pt --ref-ckpt {CKPT_DIR}/refine_best.pt
shutil.make_archive('/tmp/fp_results', 'zip', '.', 'outputs')
print('Copy box_best.pt + refine_best.pt from', CKPT_DIR, 'to floorplan_gen/checkpoints/ on your PC')